<a href="https://colab.research.google.com/github/1BM23CS345/6Sem_ML_Lab/blob/main/Lab_9_Adaboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np

# Generate synthetic data
np.random.seed(42)
num_samples = 1000

data = {
    'age': np.random.randint(18, 65, num_samples),
    'workclass': np.random.choice(['Private', 'Self-emp-not-inc', 'Federal-gov', 'Local-gov'], num_samples),
    'education': np.random.choice(['Bachelors', 'HS-grad', 'Some-college', 'Masters', 'Prof-school'], num_samples),
    'marital_status': np.random.choice(['Married-civ-spouse', 'Never-married', 'Divorced'], num_samples),
    'occupation': np.random.choice(['Adm-clerical', 'Exec-managerial', 'Handlers-cleaners', 'Prof-specialty'], num_samples),
    'hours_per_week': np.random.randint(20, 60, num_samples),
    'income': np.random.choice(['<=50K', '>50K'], num_samples, p=[0.7, 0.3]) # Target variable
}

df = pd.DataFrame(data)

# Save to a CSV file
df.to_csv('income.csv', index=False)

print("Dummy 'income.csv' created successfully with the first 5 rows:")
display(df.head())

Dummy 'income.csv' created successfully with the first 5 rows:


,age,workclass,education,marital_status,occupation,hours_per_week,income
0,56,Private,HS-grad,Never-married,Prof-specialty,20,>50K
1,46,Local-gov,HS-grad,Never-married,Prof-specialty,57,>50K
2,32,Self-emp-not-inc,HS-grad,Divorced,Adm-clerical,57,<=50K
3,60,Local-gov,HS-grad,Divorced,Prof-specialty,40,<=50K
4,25,Private,Prof-school,Divorced,Prof-specialty,48,<=50K


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load the dataset
df = pd.read_csv('income.csv')

# Separate features (X) and target (y)
X = df.drop('income', axis=1)
y = df['income'].apply(lambda x: 1 if x == '>50K' else 0) # Convert target to numerical (0 or 1)

# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

# Create preprocessing pipelines for numerical and categorical features
numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown='ignore')

# Create a column transformer to apply different transformations to different columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("Data loaded and split into training and testing sets. Preprocessing pipeline ready.")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

Data loaded and split into training and testing sets. Preprocessing pipeline ready.
X_train shape: (700, 6)
y_train shape: (700,)


In [3]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

# Define the base estimator for AdaBoost (usually a weak learner like a Decision Tree)
# A common choice is DecisionTreeClassifier with max_depth=1 (a Decision Stump)
base_estimator = DecisionTreeClassifier(max_depth=1, random_state=42)

# Create the AdaBoost Classifier pipeline
adaboost_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', AdaBoostClassifier(estimator=base_estimator, n_estimators=10, random_state=42))
])

# Train the model
adaboost_pipeline.fit(X_train, y_train)

# Make predictions on the test set
y_pred = adaboost_pipeline.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"AdaBoost Classifier (n_estimators=10) Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

AdaBoost Classifier (n_estimators=10) Accuracy: 0.7167

Classification Report:
              precision    recall  f1-score   support

           0       0.72      1.00      0.83       215
           1       0.00      0.00      0.00        85

    accuracy                           0.72       300
   macro avg       0.36      0.50      0.42       300
weighted avg       0.51      0.72      0.60       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [4]:
n_estimators_range = [5, 10, 20, 50, 100, 200, 300, 500]
best_score = 0
best_n_estimators = 0

print("\nFine-tuning AdaBoost Classifier for n_estimators:")
for n_est in n_estimators_range:
    # Create AdaBoost Classifier with current n_estimators
    adaboost_pipeline_tuned = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', AdaBoostClassifier(estimator=base_estimator, n_estimators=n_est, random_state=42))
    ])

    # Train the model
    adaboost_pipeline_tuned.fit(X_train, y_train)

    # Make predictions
    y_pred_tuned = adaboost_pipeline_tuned.predict(X_test)

    # Evaluate
    score = accuracy_score(y_test, y_pred_tuned)
    print(f"  n_estimators={n_est}: Accuracy = {score:.4f}")

    if score > best_score:
        best_score = score
        best_n_estimators = n_est

print(f"\nBest Accuracy achieved: {best_score:.4f} with n_estimators = {best_n_estimators}")


Fine-tuning AdaBoost Classifier for n_estimators:
  n_estimators=5: Accuracy = 0.7167
  n_estimators=10: Accuracy = 0.7167
  n_estimators=20: Accuracy = 0.7067
  n_estimators=50: Accuracy = 0.7100
  n_estimators=100: Accuracy = 0.7000
  n_estimators=200: Accuracy = 0.6933
  n_estimators=300: Accuracy = 0.6900
  n_estimators=500: Accuracy = 0.6833

Best Accuracy achieved: 0.7167 with n_estimators = 5
